# 04 Errors and limitations

In [ ]:
import json, os, sys
from pathlib import Path

# Work from the repository root whether the notebook is opened from notebooks/ or the root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

cfg = json.load(open("configs/dataset.json"))
OUT = Path("results/data_quality"); OUT.mkdir(parents=True, exist_ok=True)
cfg

## Error breakdowns (written by `src/04_final_test.py`)

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 7))
for a, (name, x) in zip(ax.ravel(), [("segment_id", "segment_id"), ("hour", "hour"), ("dow", "dow"), ("rain", "rain")]):
    d = pd.read_csv(f"results/breakdown_{name}.csv")
    d.plot(x=x, y=["MAE", "MAE_persistence"], kind="bar" if name != "hour" else "line", ax=a, title=f"MAE by {x}")
    a.set_ylabel("km/h")
plt.tight_layout(); plt.savefig("results/breakdowns.png", dpi=150); plt.show()

## Three representative failures

In [ ]:
t = pd.read_parquet("results/test_predictions.parquet")
t["abs_err"] = (t["target_2h"] - t["pred"]).abs()
worst_days = (t.assign(day=t["timestamp"].dt.date).groupby("day")["abs_err"].mean().nlargest(5).round(2))
print("Days with the largest mean error:\n", worst_days)
t.nlargest(10, "abs_err")[["timestamp", "segment_id", "speed_kmh", "target_2h", "pred", "abs_err", "precipitation"]].round({"speed_kmh": 1, "target_2h": 1, "pred": 1, "abs_err": 1, "precipitation": 1})

Pick three cases above. For each, write: what happened (date, segment), what the model predicted versus actual,
and a likely reason (sudden incident, holiday, rain, sensor outage). Say it is a likely reason, not a proven one.

## Limitations
1. Results describe these segments and this period only.
2. Sensor outages affect both features and target.
3. No incident, roadworks or event data.
4. Weather is observed at time t, not forecast.
5. Patterns shift across seasons and infrastructure changes.
6. Prediction is association, not causation.

Add your own specific limitations from the results (for example, anything the data-quality checks revealed).